# Retail Performance Analytics

# SQL Analysis

This notebook builds on the work completed in the previous stages of the project.

Using the cleaned dataset, the analyses presented in the previous notebook are reproduced using SQL.

The objective is to answer the same business questions directly from a relational database while demonstrating SQL skills commonly used in Business Intelligence.

## Project Roadmap

| Stage | Status |
|--------|--------|
| Business Understanding | ✅ Completed |
| Data Understanding | ✅ Completed |
| Data Cleaning | ✅ Completed |
| Business Analysis | ✅ Completed |
| SQL Analysis |  ✅ Completed |
| Power BI Dashboard | 🟡 In Progress | 
| Executive Report | ⏳ Planned |

## Notebook Objectives

This notebook aims to:

- Load the cleaned dataset into a SQLite database.
- Answer business questions using SQL queries.
- Demonstrate SQL techniques commonly used in Business Intelligence.
- Compare the SQL results with the previous Python analysis.

## Analytical Approach

Instead of using pandas for the analysis, this notebook answers the business questions using SQL.

Each section presents a SQL query, followed by the results, observations and business insights.

## Business Questions

This analysis seeks to answer the following business questions:

- How is the business performing overall?
- Who are the most valuable customers?
- Which products drive revenue?
- How does sales performance vary across countries?
- How do sales evolve over time?

In [1]:
import sqlite3

import pandas as pd

# Pandas display options
pd.set_option("display.max_columns", None)

## Load Clean Dataset

In [2]:
clean_df = pd.read_csv(
    "../data/processed/online_retail_clean.csv",
    dtype={
        "InvoiceNo": str,
        "StockCode": str
    },
    parse_dates=["InvoiceDate"]
)

#### Observation

The cleaned dataset created during the previous stage of the project is loaded into a pandas DataFrame before being imported into SQLite.

## Create SQLite Database

In [3]:
connection = sqlite3.connect(
    "../data/processed/retail.db"
)

#### Observation

A SQLite database is created to store the cleaned dataset and support the analyses presented in this notebook.

## Load Data into SQLite

In [4]:
rows_inserted = clean_df.to_sql(
    name="retail",
    con=connection,
    if_exists="replace",
    index=False
)

print(f"Rows inserted: {rows_inserted:,}")

Rows inserted: 525,460


#### Observation

The cleaned dataset was successfully imported into the **retail** table.

A total of **525,460** rows were loaded into the SQLite database and will be used throughout the SQL analysis.

## Database Schema

In [5]:
pd.read_sql(
    """
    PRAGMA table_info(retail);
    """,
    connection
)

,cid,name,type,notnull,dflt_value,pk
0,0,InvoiceNo,TEXT,0,None,0
1,1,StockCode,TEXT,0,None,0
2,2,Description,TEXT,0,None,0
3,3,Quantity,INTEGER,0,None,0
4,4,InvoiceDate,TIMESTAMP,0,None,0
5,5,UnitPrice,REAL,0,None,0
6,6,CustomerID,REAL,0,None,0
7,7,Country,TEXT,0,None,0


#### Observation

The schema confirms that the **retail** table was created successfully and that all columns were imported with the expected data types.

## Database Validation

In [6]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS TotalRows
    FROM retail;
    """,
    connection
)

,TotalRows
0,525460


#### Observation

The number of rows stored in the database matches the number of rows imported into the **retail** table.

In [7]:
pd.read_sql(
    """
    SELECT *
    FROM retail
    LIMIT 5;
    """,
    connection
)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


#### Observation

The first records confirm that the data was imported correctly and that the table is ready for the SQL analyses presented in the following sections.

# Business Question 1

## How is the business performing overall?

This section answers the first business question using SQL queries.

The analysis begins with the main business KPIs before exploring customers, products, countries and sales trends in more detail.

### SQL Query

In [8]:
pd.read_sql(
    """
    SELECT
        ROUND(SUM(Quantity * UnitPrice), 2) AS TotalSales,
        COUNT(DISTINCT InvoiceNo) AS TotalOrders,
        ROUND(
            SUM(Quantity * UnitPrice) /
            COUNT(DISTINCT InvoiceNo),
            2
        ) AS AverageOrderValue,
        COUNT(DISTINCT CustomerID) AS TotalCustomers,
        COUNT(DISTINCT StockCode) AS UniqueProducts,
        COUNT(DISTINCT Country) AS Countries
    FROM retail;
    """,
    connection
)

,TotalSales,TotalOrders,AverageOrderValue,TotalCustomers,UniqueProducts,Countries
0,10642110.8,20134,528.56,4339,3925,38


#### Observation

The SQL query returns the main business KPIs in a single result.

The values match those obtained during the Python analysis, confirming that the cleaned dataset was successfully imported into SQLite.

### Business Insight

Producing the same business metrics with SQL demonstrates that the analysis can be performed directly in a relational database.

This approach is commonly used in Business Intelligence workflows, where KPIs are calculated using SQL before being presented in reports and dashboards.

#### Consistency Check

The SQL results are consistent with the Business Analysis notebook.

# Business Question 2
## Who are the most valuable customers?

This section identifies the customers who generated the highest sales.

The analysis also considers purchase frequency to better understand customer behaviour.

In [9]:
pd.read_sql(
    """
    SELECT
        CustomerID,
        ROUND(SUM(Quantity * UnitPrice), 2) AS TotalSales,
        COUNT(DISTINCT InvoiceNo) AS Orders,
        SUM(Quantity) AS ItemsPurchased,
        ROUND(
            SUM(Quantity * UnitPrice) /
            COUNT(DISTINCT InvoiceNo),
            2
        ) AS AverageOrderValue
    FROM retail
    WHERE CustomerID IS NOT NULL
    GROUP BY CustomerID
    ORDER BY TotalSales DESC
    LIMIT 10;
    """,
    connection
)

,CustomerID,TotalSales,Orders,ItemsPurchased,AverageOrderValue
0,14646.0,280206.02,74,197491,3786.57
1,18102.0,259657.30,60,64124,4327.62
2,17450.0,194390.79,46,69973,4225.89
3,16446.0,168472.50,2,80997,84236.25
4,14911.0,143711.17,201,80490,714.98
5,12415.0,124914.53,21,77670,5948.31
6,14156.0,117210.08,55,57768,2131.09
7,17511.0,91062.38,31,64549,2937.50
8,16029.0,80850.84,63,40108,1283.35
9,12346.0,77183.60,1,74215,77183.60


#### Observation

The highest revenue is concentrated among a relatively small number of customers.

Purchase frequency and average order value vary considerably across the top customers, indicating different purchasing patterns.

### Business Insight

Combining revenue, purchase frequency and average order value provides a more complete understanding of customer behaviour.

These metrics help identify customers who may benefit from loyalty initiatives and personalised marketing strategies.

#### Consistency Check

The SQL results are consistent with the Business Analysis notebook.

# Business Question 3

## Which products drive revenue?

This section identifies the products that generated the highest sales.

The analysis considers revenue, quantity sold and the number of orders to better understand product performance.

### SQL Query

In [10]:
pd.read_sql(
    """
    SELECT
        StockCode,
        Description,
        ROUND(SUM(Quantity * UnitPrice), 2) AS TotalSales,
        SUM(Quantity) AS QuantitySold,
        COUNT(DISTINCT InvoiceNo) AS Orders
    FROM retail
    WHERE Description NOT IN (
        'DOTCOM POSTAGE',
        'POSTAGE',
        'Manual'
    )
    GROUP BY
        StockCode,
        Description
    ORDER BY TotalSales DESC
    LIMIT 10;
    """,
    connection
)

,StockCode,Description,TotalSales,QuantitySold,Orders
0,22423,REGENCY CAKESTAND 3 TIER,174156.54,13862,1989
1,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104284.24,37584,2193
3,47566,PARTY BUNTING,99445.23,18287,1686
4,85099B,JUMBO BAG RED RETROSPOT,94159.81,48375,2092
5,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,247
6,23084,RABBIT NIGHT LIGHT,66870.03,30739,994
7,22086,PAPER CHAIN KIT 50'S CHRISTMAS,64875.59,19329,1160
8,84879,ASSORTED COLOUR BIRD ORNAMENT,58927.62,36362,1455
9,79321,CHILLI LIGHTS,54096.36,10305,663


#### Observation

A relatively small number of products accounts for a large share of total sales.

The ranking also shows that high revenue is not always associated with the largest number of orders.

### Business Insight

Monitoring the performance of high-selling products supports inventory planning and commercial decisions.

Combining revenue, sales volume and order frequency provides a broader view of product performance.

#### Consistency Check

The SQL results are consistent with the Business Analysis notebook.

# Business Question 4

## How does sales performance vary across countries?

This section compares sales performance across countries.

The analysis examines revenue, order volume and the number of identified customers in each market.

### SQL Query

In [11]:
pd.read_sql(
    """
    SELECT
        Country,
        ROUND(SUM(Quantity * UnitPrice), 2) AS TotalSales,
        COUNT(DISTINCT InvoiceNo) AS Orders,
        COUNT(DISTINCT CustomerID) AS Customers
    FROM retail
    GROUP BY Country
    ORDER BY TotalSales DESC;
    """,
    connection
)

,Country,TotalSales,Orders,Customers
0,United Kingdom,9001744.09,18192,3921
1,Netherlands,285446.34,95,9
2,EIRE,283140.52,288,3
3,Germany,228678.40,457,94
4,France,209625.37,392,87
5,Australia,138453.81,57,9
6,Spain,61558.56,90,30
7,Switzerland,57067.60,54,21
8,Belgium,41196.34,98,25
9,Sweden,38367.83,36,8


#### Observation

The United Kingdom generated substantially higher sales than any other country.

The remaining markets contributed a much smaller share of total revenue.

### Business Insight

The results indicate that the business relies heavily on the United Kingdom while maintaining sales across several international markets.

Understanding the performance of these markets may help identify future growth opportunities.

#### Consistency Check

The SQL results are consistent with the Business Analysis notebook.

# Business Question 5

## How do sales evolve over time?

This section analyses how sales changed over time using SQL.

Monthly sales are calculated directly from the database to identify changes in business performance throughout the analysed period.

### SQL Query

In [12]:
pd.read_sql(
    """
    SELECT
        strftime('%Y-%m', InvoiceDate) AS Month,
        ROUND(SUM(Quantity * UnitPrice), 2) AS TotalSales,
        COUNT(DISTINCT InvoiceNo) AS Orders
    FROM retail
    GROUP BY Month
    ORDER BY Month;
    """,
    connection
)

,Month,TotalSales,Orders
0,2010-12,821452.73,1566
1,2011-01,689811.61,1089
2,2011-02,522545.56,1101
3,2011-03,716215.26,1459
4,2011-04,536968.49,1255
5,2011-05,769296.61,1689
6,2011-06,760547.01,1538
7,2011-07,718076.12,1485
8,2011-08,757841.38,1365
9,2011-09,1056435.19,1858


#### Observation

Monthly sales generally increased throughout 2011, reaching their highest level in November.

The lower sales recorded in December reflect an incomplete month rather than a confirmed decline in business performance.

### Business Insight

SQL makes it possible to analyse business trends over time directly from the database.

The results are consistent with the Python analysis and confirm the overall sales growth observed during most of 2011.

#### Consistency Check

The SQL results are consistent with the Business Analysis notebook.

## Key Findings

The SQL queries successfully reproduced the main business analyses performed in Python.

The main findings are summarised below.

- The business generated more than **£10.6 million** in sales across **20,134 completed orders**.

- Revenue is concentrated among a relatively small number of customers.

- A limited number of products generated a large share of total sales.

- The United Kingdom remains the company's primary market.

- Monthly sales increased throughout most of 2011, with November recording the highest sales.

## Conclusion

The SQL queries reproduced the main business metrics and produced results consistent with the previous Python analysis.

This demonstrates how SQL can be used to answer business questions directly from a relational database while supporting reporting and decision-making processes.

## Next Step

The next stage of the project focuses on developing an interactive Power BI dashboard.

The dashboard will combine the analyses performed in Python and SQL into a visual report designed to support business users.

In [13]:
connection.close()

print("SQLite connection closed.")

SQLite connection closed.
